# Step 1: Data Ingestion

This notebook covers testing the process of fetching the raw dataset from a Google Drive zip link, unzipping it, and loading the resulting CSV into our SQLite database in a new table called `real_bookings`.

In [1]:
# Install necessary packages if not present
%pip install gdown pandas

  Using cached gdown-5.2.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
Using cached gdown-5.2.1-py3-none-any.whl (18 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached PySocks-1.7.1-py3-none-any.whl (16 kB)

   ---------------------------------------- 0/4 [soupsieve]
   ---------------------------------------- 0/4 [soupsieve]
   ---------------------------------------- 0/4 [soupsieve]
   ---------- ----------------------------- 1/4 [PySocks]
   ---------- ----------------------------- 1/4 [PySocks]
   ---------- ----------------------------- 1/4 [PySocks]
   ---------- ----------------------------- 1/4 [PySocks]
   ---------- ----------------------------- 1/4 [PySocks]
   ---------- ----------------------------- 1/4 [PySocks]
   -------------------- ------------------- 2/4 [beautifulsoup4]
   -------------------- ------------------- 2/

In [2]:
import gdown
import zipfile
import os
import pandas as pd
import sqlite3

# 1. Setup paths
data_dir = "../data"
os.makedirs(data_dir, exist_ok=True)
zip_path = os.path.join(data_dir, "hotel_booking_data.zip")

# 2. Download from Google Drive
# File ID from the provided link
file_id = "1LYUiAnzB0UPdCAjImVdAiO_SR-Jbi8Wv"
url = f"https://drive.google.com/uc?id={file_id}"

print("Downloading dataset from Google Drive...")
gdown.download(url, zip_path, quiet=False)

# 3. Unzip the file
print("\nExtracting zip file...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(data_dir)
    extracted_files = zip_ref.namelist()
    print(f"Extracted files: {extracted_files}")

# Find the CSV file (handling potential folders in zip)
csv_filename = [f for f in extracted_files if f.endswith('.csv')]
if not csv_filename:
    # If not in namelist, just search the dir recursively
    for root, dirs, files in os.walk(data_dir):
        for file in files:
            if file.endswith('.csv'):
                csv_path = os.path.join(root, file)
                break
else:
    csv_path = os.path.join(data_dir, csv_filename[0])
    
print(f"\nTarget CSV file ready for ingestion: {csv_path}")


KeyboardInterrupt



In [ ]:
# 4. Load CSV into Pandas
print("Loading CSV into pandas DataFrame...")
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows and {len(df.columns)} columns.\n")
display(df.head())

# 5. Ingest into SQLite Database
# We will connect to the main project database (assuming it's in the project root)
db_path = "../agent_memory.db"
print(f"\nConnecting to SQLite database at {db_path}...")
conn = sqlite3.connect(db_path)

# Write the data to a new table named 'real_bookings'
print("Inserting data into table 'real_bookings'...")
df.to_sql("real_bookings", conn, if_exists="replace", index=False)

print("✅ Data insertion complete!")

# Quick verification query
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM real_bookings")
row_count = cursor.fetchone()[0]
print(f"Verification: Table 'real_bookings' has {row_count} rows.")

conn.close()